# Day 2: Exploratory Data Analysis — Gym Member Churn

Before training any model, we need to understand the data:
- What does the target look like? (class balance)
- Which features separate churners from stayers?
- Are there correlations we should know about?
- Any data quality issues?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

df = pd.read_csv('/app/data/members.csv')
print(f'Shape: {df.shape}')
df.head()

## 1. Target distribution
First question: how imbalanced is our target? This affects model choice and evaluation metrics.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Counts
churned_counts = df['churned'].value_counts()
axes[0].bar(['Retained (0)', 'Churned (1)'], churned_counts.values,
            color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Churn Count')
for i, v in enumerate(churned_counts.values):
    axes[0].text(i, v + 30, str(v), ha='center', fontweight='bold')

# Rate
churn_rate = df['churned'].mean()
axes[1].bar(['Retained', 'Churned'], [1 - churn_rate, churn_rate],
            color=['#2ecc71', '#e74c3c'])
axes[1].set_title(f'Churn Rate: {churn_rate:.1%}')
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

print(f'Churn rate: {churn_rate:.1%} — moderately imbalanced.')
print(f'A model that always predicts "retained" would be {1-churn_rate:.1%} accurate.')
print(f'So our model must beat {1-churn_rate:.1%} to be useful.')

## 2. Churn rate by categorical features
Which segments churn more? This is the most actionable view for a business stakeholder.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

# By membership type
churn_by_membership = df.groupby('membership_type')['churned'].agg(['mean', 'count'])
churn_by_membership = churn_by_membership.sort_values('mean', ascending=False)
bars = axes[0].bar(churn_by_membership.index, churn_by_membership['mean'],
                    color=['#e74c3c', '#f39c12', '#2ecc71'])
axes[0].set_title('Churn Rate by Membership Type')
axes[0].set_ylim(0, 0.35)
for bar, (_, row) in zip(bars, churn_by_membership.iterrows()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{row["mean"]:.1%}\n(n={int(row["count"])})',
                 ha='center', fontsize=9)

# By referral
churn_by_ref = df.groupby('has_referral')['churned'].mean()
axes[1].bar(['No Referral', 'Has Referral'], churn_by_ref.values,
            color=['#e74c3c', '#2ecc71'])
axes[1].set_title('Churn Rate by Referral')
axes[1].set_ylim(0, 0.35)
for i, v in enumerate(churn_by_ref.values):
    axes[1].text(i, v + 0.005, f'{v:.1%}', ha='center', fontweight='bold')

# By gender
churn_by_gender = df.groupby('gender')['churned'].mean()
axes[2].bar(churn_by_gender.index, churn_by_gender.values,
            color=['#3498db', '#e91e63'])
axes[2].set_title('Churn Rate by Gender')
axes[2].set_ylim(0, 0.35)
for i, v in enumerate(churn_by_gender.values):
    axes[2].text(i, v + 0.005, f'{v:.1%}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Numeric feature distributions: churners vs retained
The key question: do the distributions SEPARATE? If churners look different from retained members on a feature, that feature will be useful for the model.

In [ ]:
numeric_features = [
    'visits_last_30d', 'days_since_last_visit', 'visit_trend',
    'tenure_months', 'late_payments', 'pct_morning_visits',
    'distance_miles', 'unique_class_types', 'avg_visits_per_week'
]

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i, feat in enumerate(numeric_features):
    ax = axes[i]
    retained = df[df['churned'] == 0][feat]
    churned = df[df['churned'] == 1][feat]
    
    ax.hist(retained, bins=30, alpha=0.6, color='#2ecc71', label='Retained', density=True)
    ax.hist(churned, bins=30, alpha=0.6, color='#e74c3c', label='Churned', density=True)
    ax.set_title(feat)
    ax.legend(fontsize=8)

plt.suptitle('Feature Distributions: Churned vs Retained', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print('Look for features where red and green distributions SEPARATE.')
print('More separation = more predictive power for the model.')

## 4. Median comparison: churners vs retained
A quick numerical summary of the differences.

In [ ]:
comparison = df.groupby('churned')[numeric_features].median().T
comparison.columns = ['Retained (median)', 'Churned (median)']
comparison['Difference'] = comparison['Churned (median)'] - comparison['Retained (median)']
comparison['Direction'] = comparison['Difference'].apply(
    lambda x: '↑ churners higher' if x > 0 else '↓ churners lower'
)
comparison.style.background_gradient(subset=['Difference'], cmap='RdYlGn_r')

## 5. Correlation heatmap
Which features are correlated with each other? Highly correlated features may be redundant — a model doesn't need both.

In [ ]:
corr_cols = numeric_features + ['churned']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, ax=ax,
            vmin=-1, vmax=1)
ax.set_title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

print('\nCorrelation with churn target (absolute value, ranked):')
churn_corr = corr['churned'].drop('churned').abs().sort_values(ascending=False)
for feat, val in churn_corr.items():
    direction = '+' if corr.loc[feat, 'churned'] > 0 else '-'
    print(f'  {feat:25s}  {direction}{val:.3f}')

## 6. Churn rate by visit bins
Visit frequency is usually the #1 churn predictor in fitness. Let's verify.

In [ ]:
df['visit_bin'] = pd.cut(df['visits_last_30d'],
                          bins=[-1, 0, 2, 5, 10, 15, 30],
                          labels=['0', '1-2', '3-5', '6-10', '11-15', '16+'])

visit_churn = df.groupby('visit_bin', observed=True).agg(
    churn_rate=('churned', 'mean'),
    count=('churned', 'count')
).reset_index()

fig, ax1 = plt.subplots(figsize=(10, 5))

bars = ax1.bar(visit_churn['visit_bin'], visit_churn['churn_rate'],
               color=['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#27ae60', '#1a9850'])
ax1.set_ylabel('Churn Rate')
ax1.set_xlabel('Visits in Last 30 Days')
ax1.set_title('Churn Rate by Recent Visit Frequency')

for bar, (_, row) in zip(bars, visit_churn.iterrows()):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{row["churn_rate"]:.0%}\n(n={row["count"]})',
             ha='center', fontsize=9)

ax1.set_ylim(0, max(visit_churn['churn_rate']) * 1.3)
plt.tight_layout()
plt.show()

# Clean up temp column
df.drop(columns=['visit_bin'], inplace=True)

## 7. Tenure vs churn
New members churn more — but how much more?

In [ ]:
df['tenure_bin'] = pd.cut(df['tenure_months'],
                           bins=[0, 3, 6, 12, 24, 72],
                           labels=['0-3mo', '3-6mo', '6-12mo', '1-2yr', '2yr+'])

tenure_churn = df.groupby('tenure_bin', observed=True).agg(
    churn_rate=('churned', 'mean'),
    count=('churned', 'count')
).reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(tenure_churn['tenure_bin'], tenure_churn['churn_rate'],
              color=['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#1a9850'])
ax.set_ylabel('Churn Rate')
ax.set_xlabel('Member Tenure')
ax.set_title('Churn Rate by Tenure')

for bar, (_, row) in zip(bars, tenure_churn.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{row["churn_rate"]:.0%}\n(n={row["count"]})',
             ha='center', fontsize=9)

ax.set_ylim(0, max(tenure_churn['churn_rate']) * 1.3)
plt.tight_layout()
plt.show()

df.drop(columns=['tenure_bin'], inplace=True)

## 8. Summary statistics

In [ ]:
print('=== Data Quality ===')
print(f'Rows: {len(df)}')
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Duplicate member_ids: {df["member_id"].duplicated().sum()}')
print()
print('=== Feature Types ===')
print(f'Numeric: {len(df.select_dtypes(include=["number"]).columns)}')
print(f'Categorical: {len(df.select_dtypes(include=["object"]).columns)}')
print()
df.describe().round(2)

## Key Takeaways

After running all cells above, you should see:

1. **~18% churn rate** — imbalanced but workable. We'll use precision/recall/AUC, not just accuracy.
2. **Visit frequency is the #1 signal** — members with 0-2 visits in last 30 days churn at much higher rates.
3. **Tenure matters** — new members (0-3 months) churn far more than veterans.
4. **Basic membership = higher churn** — likely price-sensitive or less committed.
5. **Referrals help** — social accountability reduces churn.
6. **Visit features are correlated** — visits_last_30d, visits_last_60d, avg_visits_per_week overlap. We'll handle this in feature engineering.

**Next step (Day 3):** Feature engineering — create better features from these raw ones, handle the correlated visit features, encode categoricals.